<h2>使用 Selenium 自动化抓取</h2>

JavaScript 动态渲染的页面不止 Ajax 这一种。比如中国青年网,  https://news.youth.cn/gn/ 它的分页部分是由 JavaScript 生成的，并非原始 HTML 代码，这其中并不包含 Ajax 请求。

它即使是 Ajax 获取的数据，但是其 Ajax 接口含有很多加密参数，难以直接找出其规律，也很难直接分析 Ajax 来抓取。

为了解决这些问题，可以**直接使用模拟浏览器运行的方式来实现**，这样就可以做到在浏览器中看到是什么样，抓取的源码就是什么样，也就是可见即可爬。

Python 提供了许多模拟浏览器运行的库，如 `Selenium`、`Splash`、`PyV8`、`Ghost` 等。

<h3>Selenium 的使用</h3>

`Selenium` 是一个自动化测试工具，利用它可以驱动浏览器执行特定的动作，如点击、下拉等操作，同时还可以获取浏览器当前呈现的页面的源代码，做到可见即可爬。

以 Chrome 为例来讲解 `Selenium` 的用法。
> 在开始之前，确保已经正确安装好了 Chrome 浏览器并配置好了 ChromeDriver；还需要正确安装好 Python 的 `Selenium` 库。
>
> 运行代码后发现，会自动弹出一个 Chrome 浏览器。浏览器首先会跳转到百度，然后在搜索框中输入 Python，接着跳转到搜索结果页。

In [3]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.wait import WebDriverWait

browser = webdriver.Chrome()
try:
    browser.get('https://www.baidu.com')
    # 在Selenium 4之后的版本中，find_element_by_id() 方法已被弃用
    input = browser.find_element(By.ID, 'kw')
    input.send_keys('Python')
    input.send_keys(Keys.ENTER)
    wait = WebDriverWait(browser, 10)
    wait.until(EC.presence_of_element_located((By.ID, 'content_left')))
    print(browser.current_url)
    print(browser.get_cookies())
    # print(browser.page_source)
finally:
    browser.close()

The chromedriver version (126.0.6478.63) detected in PATH at D:\Python\Anaconda\Scripts\chromedriver.exe might not be compatible with the detected chrome version (127.0.6533.100); currently, chromedriver 127.0.6533.99 is recommended for chrome 127.*, so it is advised to delete the driver in PATH and retry


https://www.baidu.com/s?ie=utf-8&f=8&rsv_bp=1&rsv_idx=1&tn=baidu&wd=Python&fenlei=256&rsv_pq=0xd70e3dab00534355&rsv_t=605fkm1LZyPQM5chOtD%2Fa90PekV02Meml6lmm8FRLvvIVPHlTsRIMuwJlnpZ&rqlang=en&rsv_enter=1&rsv_dl=tb&rsv_sug3=6&rsv_sug2=0&rsv_btype=i&inputT=63&rsv_sug4=63
[{'domain': 'www.baidu.com', 'expiry': 1754833726, 'httpOnly': False, 'name': 'COOKIE_SESSION', 'path': '/', 'sameSite': 'Lax', 'secure': False, 'value': '0_0_0_1_0_1_0_0_0_1_2_0_0_0_0_0_0_0_1723297729%7C1%230_0_1723297729%7C1'}, {'domain': '.baidu.com', 'httpOnly': False, 'name': 'delPer', 'path': '/', 'sameSite': 'Lax', 'secure': False, 'value': '0'}, {'domain': '.baidu.com', 'expiry': 1754833725, 'httpOnly': False, 'name': 'H_PS_PSSID', 'path': '/', 'sameSite': 'Lax', 'secure': False, 'value': '60276_60522_60566_60555'}, {'domain': '.baidu.com', 'httpOnly': False, 'name': 'PSINO', 'path': '/', 'sameSite': 'Lax', 'secure': False, 'value': '5'}, {'domain': '.baidu.com', 'expiry': 1723384127, 'httpOnly': False, 'name': 'B

**1. 声明浏览器对象（初始化）**

Selenium 支持非常多的浏览器，如 Chrome、Firefox、Edge 等，还有 Android、BlackBerry 等手机端的浏览器。另外，也支持无界面浏览器 PhantomJS。

```python
from selenium import webdriver

browser = webdriver.Chrome()
browser = webdriver.Firefox()
browser = webdriver.Edge()
browser = webdriver.PhantomJS()
browser = webdriver.Safari()
```

这样就完成了浏览器对象的初始化并将其赋值为 browser 对象。

**2. 调用 browser 对象，让其执行各个动作以模拟浏览器操作**

可以用 `get()` 方法来请求网页，参数传入链接 URL 即可。

```python
browser.get('https://www.taobao.com')
print(browser.page_source)
browser.close()
```

**3. 查找节点**

比如，想要从淘宝页面中提取搜索框这个节点:

![](7-1.png)

> 它的 `id` 是 `q`，`name` 也是 `q`。此外，还有许多其他属性，此时我们就可以用多种方式获取它了。
> 
> 比如，`find_element(By.NAME, 'q')` 是根据 name 值获取，`find_element(By.ID, 'q')` 是根据 id 获取。另外，还有根据 XPath、CSS 选择器等获取的方式。

In [4]:
from selenium import webdriver

browser = webdriver.Chrome()
browser.get('https://www.taobao.com')
input_first = browser.find_element(By.ID, 'q')
input_second = browser.find_element(By.CSS_SELECTOR, '#q')
input_third = browser.find_element(By.XPATH, '//*[@id="q"]')
print(input_first, input_second, input_third)
browser.close()

The chromedriver version (126.0.6478.63) detected in PATH at D:\Python\Anaconda\Scripts\chromedriver.exe might not be compatible with the detected chrome version (127.0.6533.100); currently, chromedriver 127.0.6533.99 is recommended for chrome 127.*, so it is advised to delete the driver in PATH and retry


<selenium.webdriver.remote.webelement.WebElement (session="64a5ee5f5affe0c2421df2a296433f2c", element="f.28F87733686B50314CAEAD95624859AF.d.65CDA2D666F2E7F1FE8D4D7A84917B75.e.9")> <selenium.webdriver.remote.webelement.WebElement (session="64a5ee5f5affe0c2421df2a296433f2c", element="f.28F87733686B50314CAEAD95624859AF.d.65CDA2D666F2E7F1FE8D4D7A84917B75.e.9")> <selenium.webdriver.remote.webelement.WebElement (session="64a5ee5f5affe0c2421df2a296433f2c", element="f.28F87733686B50314CAEAD95624859AF.d.65CDA2D666F2E7F1FE8D4D7A84917B75.e.9")>


**ps. 多个节点**

如果要查找所有满足条件的节点，需要用 `find_elements()` 这样的方法。

例如，要查找淘宝左侧导航条的所有条目：

![](7-2.png)

就可以这样来实现：

In [5]:
from selenium import webdriver

browser = webdriver.Chrome()
browser.get('https://www.taobao.com')
lis = browser.find_elements(By.CSS_SELECTOR, '.service-bd--B9l1TEHT li')
print(lis)
browser.close()

The chromedriver version (126.0.6478.63) detected in PATH at D:\Python\Anaconda\Scripts\chromedriver.exe might not be compatible with the detected chrome version (127.0.6533.100); currently, chromedriver 127.0.6533.99 is recommended for chrome 127.*, so it is advised to delete the driver in PATH and retry


[<selenium.webdriver.remote.webelement.WebElement (session="e17984c1a617a992726dbf8781684d96", element="f.4E5B35DB5A404617D711CC99F5DD57F4.d.1047A2321E4B4219844F860B82584F9C.e.80")>, <selenium.webdriver.remote.webelement.WebElement (session="e17984c1a617a992726dbf8781684d96", element="f.4E5B35DB5A404617D711CC99F5DD57F4.d.1047A2321E4B4219844F860B82584F9C.e.81")>, <selenium.webdriver.remote.webelement.WebElement (session="e17984c1a617a992726dbf8781684d96", element="f.4E5B35DB5A404617D711CC99F5DD57F4.d.1047A2321E4B4219844F860B82584F9C.e.82")>, <selenium.webdriver.remote.webelement.WebElement (session="e17984c1a617a992726dbf8781684d96", element="f.4E5B35DB5A404617D711CC99F5DD57F4.d.1047A2321E4B4219844F860B82584F9C.e.83")>, <selenium.webdriver.remote.webelement.WebElement (session="e17984c1a617a992726dbf8781684d96", element="f.4E5B35DB5A404617D711CC99F5DD57F4.d.1047A2321E4B4219844F860B82584F9C.e.84")>, <selenium.webdriver.remote.webelement.WebElement (session="e17984c1a617a992726dbf8781684d

**ps. 获取Cookies**

- 先登录 taobao
- 使用 `get_cookies()`获取 cookie

```python
from selenium import webdriver

browser = webdriver.Chrome()
browser.get('https://www.taobao.com/?spm=a21n57.1.logo.1.5b0a523cXbEAho')

taobao_cookies = browser.get_cookies()

for cookie in taobao_cookies:
    print("%s -> %s" % (cookie['name'], cookie['value']))
```

**5. 节点交互**

`Selenium` 可以驱动浏览器来执行一些操作。

比较常见的用法有：输入文字时用 `send_keys` 方法，清空文字时用 `clear` 方法，点击按钮时用 `click` 方法。示例如下：

In [12]:
from selenium import webdriver
import time

browser = webdriver.Chrome()
browser.get('https://www.taobao.com')

# browser.add_cookie(taobao_cookie)
# browser.refresh()

input = browser.find_element(By.ID, 'q')
input.send_keys('iPhone')
time.sleep(1)
input.clear()
input.send_keys('iPad')

button = browser.find_element(By.CLASS_NAME, 'btn-search')
button.click()

The chromedriver version (126.0.6478.63) detected in PATH at D:\Python\Anaconda\Scripts\chromedriver.exe might not be compatible with the detected chrome version (127.0.6533.100); currently, chromedriver 127.0.6533.99 is recommended for chrome 127.*, so it is advised to delete the driver in PATH and retry


更多的操作可以参见官方文档的交互动作介绍：https://selenium-python.readthedocs.io/api.html#module-selenium.webdriver.remote.webelement 